# Part B — Directed Link Prediction (25 pts) + Bonus: Future Link Prediction (+5 pts)

**Network Analysis assignment.** This notebook predicts *directed* links in a
real social network: which subreddit will hyperlink to which other subreddit.

### What is "link prediction", in plain words?
Imagine a graph (a network) is a set of dots ("nodes") connected by arrows
("directed edges"). **Link prediction** asks a simple question: *given the
arrows we can already see, can we guess which arrows are missing or will appear
next?* We turn this into a yes/no classification problem: for a candidate pair
of nodes `(u, v)`, predict whether the arrow `u -> v` should exist.

Because our network is **directed**, the arrow `u -> v` is different from
`v -> u`. A good predictor therefore has to pay attention to *direction*, not
just "are these two nodes close".

### Roadmap of this notebook (mirrors the assignment tasks)
- **B1.1** Describe the network (size, directedness, timestamps, degree
  distributions, reciprocity, connected components).
- **B1.2** Define *positive* examples (real edges) and *negative* examples
  (sampled non-edges).
- **B1.3** Split edges into a **train** set and a **test** set, and explain how
  we avoid "leakage" (cheating by peeking at the answers).
- **B1.4** A **baseline** classifier built only from *directed topology
  features* (numbers we compute from the arrow structure).
- **B1.5** An **improved** classifier that adds **node2vec embeddings**
  (learned coordinate vectors for each node).
- **B1.6** Evaluate every model with AUC, accuracy, precision, recall, F1, and
  draw ROC curves.
- **B1.7** Compare baseline vs improved.
- **Bonus** Predict *future* links using a *temporal* split (train on the past,
  test on the future) and compare against a random guesser.

Every task ends with a **"How I solved this task"** box in plain English.


## Setup: imports, seeds, headless plotting

We fix the random seeds to `42` so the notebook is **reproducible** — running it
again gives the same numbers. We also force matplotlib to use the **"Agg"
backend**, which means "draw images to files, never try to open a window". This
matters because we run on a headless cluster node that has no screen.


In [1]:
import os, sys, time, math, random, pickle, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import networkx as nx

import matplotlib
matplotlib.use("Agg")          # headless: write figures to files, never open a window
import matplotlib.pyplot as plt

# shared helpers provided by the assignment (save_fig, set_style)
sys.path.insert(0, "/home/mickaelz/Network analysis/src")
import na_utils as na
na.set_style()

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA = "/home/mickaelz/Network analysis/data/linkpred/soc-redditHyperlinks-body.tsv"
FIGDIR = "/home/mickaelz/Network analysis/figures"
print("Libraries: networkx", nx.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
print("Reproducibility: all seeds set to", RANDOM_SEED)


Libraries: networkx 3.2.1 | numpy 1.25.2 | pandas 2.2.3
Reproducibility: all seeds set to 42


## B1.1 — Network description (2 pts)

### Data source
**SNAP Reddit Hyperlink Network (body version).**
Download page: https://snap.stanford.edu/data/soc-RedditHyperlinks.html
Direct file: https://snap.stanford.edu/data/soc-redditHyperlinks-body.tsv

This dataset records, for two and a half years, every time one **subreddit** (a
topic-based community on the website Reddit) posts a **hyperlink** that points
into another subreddit. Each row is one such hyperlink event.

### What the nodes and edges mean
- **Node** = a subreddit (an online community), e.g. `askreddit`, `nfl`.
- **Directed edge** `u -> v` = community `u` posted a hyperlink that points to
  community `v`. Direction matters: "askreddit links to nfl" is not the same as
  "nfl links to askreddit".

The raw file has one row per *hyperlink post*, so the same pair `u -> v` can
appear many times. We **aggregate** these parallel events into a single
directed edge and remember three things about it:
- `weight` = how many times `u` linked to `v` (a strength measure),
- `first` = the timestamp of the *first* time `u -> v` happened,
- `last`  = the timestamp of the *last* time it happened.

The `first`/`last` timestamps are what make the **bonus** (future prediction)
possible: we can literally train on the past and test on the future.

### Why we sample (and how)
The full graph has ~35,800 subreddits and ~137,800 directed edges. Running the
embedding step (node2vec) on the whole thing on a *shared 4-CPU node* would be
slow and memory-hungry. So we keep the **subgraph induced by the top-N most
active subreddits** (N = 2000), where "active" = total number of times a
subreddit appears as either a source or a target. This is a principled sample:
the most active communities are exactly the ones with enough signal to learn
from, and restricting to them keeps the graph dense and connected instead of a
cloud of one-off links. We will report exactly how much of the data this keeps.


In [2]:
# Load only the columns we need.
df = pd.read_csv(DATA, sep="\t",
                 usecols=["SOURCE_SUBREDDIT", "TARGET_SUBREDDIT", "TIMESTAMP", "LINK_SENTIMENT"])
df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"])
df = df[df["SOURCE_SUBREDDIT"] != df["TARGET_SUBREDDIT"]]   # drop self-loops (a node linking to itself)

print("Raw hyperlink events:", len(df))
print("Full time range:", df['TIMESTAMP'].min(), "->", df['TIMESTAMP'].max())
print("Distinct subreddits (full):", len(set(df['SOURCE_SUBREDDIT']) | set(df['TARGET_SUBREDDIT'])))

# Pick the TOP_N most active subreddits and keep only events fully inside that set.
TOP_N = 2000
activity = pd.concat([df["SOURCE_SUBREDDIT"], df["TARGET_SUBREDDIT"]]).value_counts()
keep = set(activity.head(TOP_N).index)
sub = df[df["SOURCE_SUBREDDIT"].isin(keep) & df["TARGET_SUBREDDIT"].isin(keep)].copy()
print(f"\nKept {len(sub)} events ({100*len(sub)/len(df):.1f}% of all events) "
      f"among the top {TOP_N} subreddits")


Raw hyperlink events: 286561
Full time range: 2013-12-31 16:39:58 -> 2017-04-30 16:58:21
Distinct subreddits (full): 35776

Kept 155088 events (54.1% of all events) among the top 2000 subreddits


In [3]:
# Aggregate the parallel events into one directed edge per (source, target) pair.
agg = (sub.groupby(["SOURCE_SUBREDDIT", "TARGET_SUBREDDIT"])
          .agg(first=("TIMESTAMP", "min"),
               last=("TIMESTAMP", "max"),
               weight=("TIMESTAMP", "size"))
          .reset_index())

# Build the DIRECTED graph (DiGraph = a graph whose edges have a direction).
G = nx.from_pandas_edgelist(agg, "SOURCE_SUBREDDIT", "TARGET_SUBREDDIT",
                            edge_attr=["weight", "first", "last"], create_using=nx.DiGraph)
print("SAMPLED DIRECTED GRAPH")
print("  nodes (subreddits):", G.number_of_nodes())
print("  directed edges     :", G.number_of_edges())
print("  is directed?       :", G.is_directed())
print("  time range of edges:", agg['first'].min(), "->", agg['last'].max())


SAMPLED DIRECTED GRAPH
  nodes (subreddits): 1995
  directed edges     : 48998
  is directed?       : True
  time range of edges: 2013-12-31 16:39:58 -> 2017-04-30 16:54:08


In [4]:
# Basic directed statistics.
indeg  = dict(G.in_degree())    # in-degree(v)  = number of communities that link TO v
outdeg = dict(G.out_degree())   # out-degree(u) = number of communities that u links to

reciprocity = nx.reciprocity(G)             # fraction of edges u->v that also have v->u
n_scc = nx.number_strongly_connected_components(G)
n_wcc = nx.number_weakly_connected_components(G)

print("Average in-degree :", round(np.mean(list(indeg.values())), 2))
print("Average out-degree:", round(np.mean(list(outdeg.values())), 2))
print("Max in-degree     :", max(indeg.values()), "(most linked-TO community)")
print("Max out-degree    :", max(outdeg.values()), "(most linking-OUT community)")
print()
print("Reciprocity                :", round(reciprocity, 4))
print("# strongly connected comps :", n_scc)
print("# weakly  connected comps  :", n_wcc)


Average in-degree : 24.56
Average out-degree: 24.56
Max in-degree     : 850 (most linked-TO community)
Max out-degree    : 773 (most linking-OUT community)

Reciprocity                : 0.2585
# strongly connected comps : 107
# weakly  connected comps  : 2


**Reading these numbers (plain English).**

- **In-degree of a node `v`** counts how many *different* communities link *to*
  `v`. **Out-degree of `u`** counts how many different communities `u` links
  *to*. A small worked example: if only `a -> c` and `b -> c` exist, then `c`
  has in-degree 2 and out-degree 0; `a` and `b` each have out-degree 1.
- **Reciprocity** = the share of arrows that are "mutual". If `u -> v` exists,
  how often does `v -> u` also exist? A reciprocity of `0.26` means roughly a
  quarter of links are answered back. (A common misconception: reciprocity is
  *not* "how connected the graph is" — it is specifically about mutual arrows.)
- **Strongly connected component (SCC)** = a group of nodes where you can travel
  from any node to any other *following the arrow directions*. **Weakly
  connected component (WCC)** = same idea but you are allowed to walk *against*
  arrows too (treat them as plain lines). Many SCCs but very few WCCs is the
  classic shape of a directed social graph: one big "blob" of communities that
  are all reachable if you ignore direction, but lots of little one-way pockets
  when you must respect direction.


### Degree distribution plots
A **degree distribution** is a histogram answering "how many nodes have degree
1, degree 2, degree 3, ...?". Real social networks are usually **heavy-tailed**:
most nodes have a tiny degree, but a few "hubs" have an enormous degree. We plot
in-degree and out-degree on **log-log axes** (both axes spaced by powers of ten)
because that squashes the huge range into a readable picture; a roughly straight
line on log-log axes is the signature of a heavy-tailed ("scale-free-like")
network.

In [5]:
def degree_hist(degrees):
    vals = np.array([d for d in degrees.values() if d > 0])
    counts = np.bincount(vals)
    ks = np.nonzero(counts)[0]
    return ks, counts[ks]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (deg, name, color) in zip(axes,
        [(indeg, "In-degree", "#1f77b4"), (outdeg, "Out-degree", "#d62728")]):
    ks, cs = degree_hist(deg)
    ax.scatter(ks, cs, s=14, color=color, alpha=0.7)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(name + " (k)"); ax.set_ylabel("number of nodes with degree k")
    ax.set_title(f"{name} distribution (log-log)")
fig.suptitle("B1.1 Degree distributions of the Reddit hyperlink subgraph", y=1.02)
fig.tight_layout()
na.save_fig(fig, "partB_degree_distributions.png")
plt.close(fig)
print("saved figures/partB_degree_distributions.png")


saved figures/partB_degree_distributions.png


**How I solved this task (B1.1).**
I downloaded the SNAP Reddit hyperlink "body" file (one row per hyperlink post,
with timestamps). I aggregated repeated `u -> v` posts into single directed
edges, keeping a weight (how many posts) and the first/last timestamp. Because
the full graph is large for embedding on a shared CPU node, I kept the subgraph
induced by the 2000 most active subreddits, which still retains over half of all
link events while staying dense and (almost entirely) connected. I then reported
the standard directed descriptors — in/out-degree averages and maxima,
reciprocity, and the number of strongly and weakly connected components — and
plotted the in- and out-degree distributions on log-log axes. The main finding
is that this is a typical heavy-tailed directed social graph: a few hub
communities dominate, about a quarter of links are reciprocated, and there is
essentially one giant weakly connected blob but many small strongly connected
pockets.

## B1.2 — Positive and negative examples (3 pts)

A classifier learns from **labelled examples**: inputs tagged with the right
answer. For link prediction the two labels are:

- **Positive example (label 1)** = a *real* directed edge `u -> v` that exists
  in the graph. This is a pair that genuinely should be linked.
- **Negative example (label 0)** = a *non-edge* `u -> v`, i.e. an ordered pair
  of nodes with **no** arrow from `u` to `v`. This is a pair that should *not*
  be linked.

### Why we must *sample* negatives, and how we sample them
In a graph with ~2000 nodes there are about `2000 x 1999 = 4 million` possible
ordered pairs, but only ~49,000 are real edges. So **non-edges hugely outnumber
edges**. If we used *all* non-edges, the data would be ~99% negatives and a lazy
model could score 99% accuracy by always saying "no link" — useless. We
therefore **balance** the classes: we randomly sample exactly as many negatives
as we have positives (a 50/50 mix).

Two rules make the negatives *fair* rather than trivially easy:
1. **Only active nodes.** Both `u` and `v` must be in our top-2000 sample, so we
   never invent a pair involving a community we know nothing about.
2. **Same weakly connected component.** We require `u` and `v` to be in the same
   WCC (reachable if you ignore arrow direction). This rules out
   *trivially-disconnected* pairs — two communities in totally separate islands
   of the graph, which any method could reject without learning anything. Forcing
   `u` and `v` into the same blob makes the negative "look plausible", so the
   classifier has to actually use structure to tell real from fake.


In [6]:
nodes = list(G.nodes())
full_edge_set = set(G.edges())          # every real directed edge (used as "forbidden")

# WCC membership, for the "same component" filter.
wcc_of = {}
for ci, comp in enumerate(nx.weakly_connected_components(G)):
    for nd in comp:
        wcc_of[nd] = ci

# sample_negatives: draw directed non-edges (u,v) with u!=v, (u,v) not a real
# edge, and u,v in the same weakly connected component (avoids trivial pairs).
def sample_negatives(n_needed, forbidden_edges, seed):
    r = np.random.RandomState(seed)
    arr = np.array(nodes, dtype=object)
    negs = set()
    while len(negs) < n_needed:
        us = arr[r.randint(0, len(arr), size=n_needed * 2)]
        vs = arr[r.randint(0, len(arr), size=n_needed * 2)]
        for u, v in zip(us, vs):
            if u == v or (u, v) in forbidden_edges or (u, v) in negs:
                continue
            if wcc_of.get(u) != wcc_of.get(v):
                continue                 # skip trivially-disconnected pairs
            negs.add((u, v))
            if len(negs) >= n_needed:
                break
    return list(negs)

n_pos = G.number_of_edges()
neg_all = sample_negatives(n_pos, full_edge_set, seed=RANDOM_SEED)
print("positive examples (real directed edges):", n_pos)
print("negative examples (sampled non-edges)  :", len(neg_all))
print("class balance: 50% / 50%")


positive examples (real directed edges): 48998
negative examples (sampled non-edges)  : 48998
class balance: 50% / 50%


**How I solved this task (B1.2).**
I labelled all real directed edges as positives. For negatives I randomly drew
ordered pairs `(u, v)` that are not real edges, are not self-loops, and lie in
the same weakly connected component, sampling exactly as many negatives as there
are positives to get a balanced 50/50 dataset. The "same component" filter is
the key design choice: it removes pairs that are trivially unlinkable (different
islands of the graph) so the classifier is forced to learn from real structural
signal rather than from an easy giveaway.

## B1.3 — Train/test split and leakage avoidance (3 pts)

To judge a model honestly we hide some examples during training and only reveal
them at test time. We use a **random 80/20 edge split**: 80% of the real edges
(and 80% of the negatives) become the **training set**; the remaining 20% become
the **test set**.

### What "leakage" is and how we prevent it
**Leakage** means the model accidentally sees information about the test answers
while training — like a student who studied the exam beforehand. The subtle
danger here is in *feature computation*. We describe each candidate pair `(u, v)`
with numbers computed from the graph (degrees, shared neighbours, paths, ...).
If we computed those numbers from the **full** graph, then for a test edge
`u -> v` the graph would still *contain that edge*, and features like "do `u`
and `v` share neighbours" would secretly encode the answer.

**Our fix:** we build a separate **train graph** `G_train` that contains *only
the training positive edges*. Every feature — for both train and test pairs — is
computed from `G_train` alone. A test edge `u -> v` is therefore *absent* from
the graph we measure, exactly as a truly unknown future link would be. This is
the standard, leakage-free way to set up link prediction.

(For the bonus we will use a different, *temporal* split — train on edges before
a cutoff date, test on edges after it — which is even more realistic.)


In [7]:
edges = list(G.edges())
rng = np.random.RandomState(RANDOM_SEED)
perm = rng.permutation(len(edges))
n_test = int(0.2 * len(edges))
test_idx = set(perm[:n_test].tolist())

test_pos  = [edges[i] for i in range(len(edges)) if i in test_idx]
train_pos = [edges[i] for i in range(len(edges)) if i not in test_idx]

# The TRAIN graph: only training positive edges. ALL features come from this.
G_train = nx.DiGraph()
G_train.add_nodes_from(G.nodes())                 # keep all nodes so isolated nodes still exist
for u, v in train_pos:
    G_train.add_edge(u, v, **G[u][v])

# Split the negatives the same 80/20 way.
perm_n = rng.permutation(len(neg_all))
neg_test  = [neg_all[i] for i in perm_n[:n_test]]
neg_train = [neg_all[i] for i in perm_n[n_test:]]

print("train positives:", len(train_pos), "| test positives:", len(test_pos))
print("train negatives:", len(neg_train), "| test negatives:", len(neg_test))
print("G_train edges (features come ONLY from here):", G_train.number_of_edges())


train positives: 39199 | test positives: 9799
train negatives: 39199 | test negatives: 9799
G_train edges (features come ONLY from here): 39199


**How I solved this task (B1.3).**
I randomly assigned 20% of the real edges to a held-out test set and 80% to
training, and split the sampled negatives the same way. Crucially, I rebuilt a
*train-only* directed graph that contains just the training edges and compute
every feature from that graph. This guarantees a test edge is invisible while we
describe it, so the evaluation reflects genuine prediction rather than memorised
answers — the textbook way to avoid leakage.

## B1.4 — Baseline classifier from directed topology features (7 pts)

"**Topology**" just means "the shape of the graph" — who points to whom. A
"**feature**" is a single number we measure for a candidate pair `(u, v)`. The
classifier sees a row of such numbers and outputs a probability that `u -> v`
should exist. Because the graph is directed, our features explicitly use
*direction*. Below, each feature is defined in plain words with a tiny example.

Let `succ(x)` = the set of nodes `x` **points to** (its out-neighbours), and
`pred(x)` = the set of nodes that **point to** `x` (its in-neighbours).

1. **`src_out`, `src_in`, `tgt_out`, `tgt_in`** — the out- and in-degrees of the
   source `u` and target `v`. (Raw "how busy is each endpoint" signals.)
2. **`common_succ`** — how many nodes *both* `u` and `v` point to,
   `|succ(u) ∩ succ(v)|`. If `u` and `v` link to many of the same communities
   they "behave alike", a hint they might link to each other.
3. **`common_pred`** — how many nodes point to *both* `u` and `v`,
   `|pred(u) ∩ pred(v)|`.
4. **`dir_jaccard_succ`** — **Jaccard similarity** of their successor sets:
   `|succ(u) ∩ succ(v)| / |succ(u) ∪ succ(v)|`. Jaccard = "share of things they
   have in common out of everything either has". It ranges 0 (nothing shared) to
   1 (identical sets). Example: if `succ(u)={a,b}` and `succ(v)={b,c}`, Jaccard
   = 1 shared / 3 total = 0.33.
5. **`dir_jaccard_pred`** — same Jaccard idea but on predecessor sets.
6. **`dir_adamic_adar`** — **directed Adamic-Adar**. We look at "stepping-stone"
   nodes `w` on a path `u -> w -> v` (so `w ∈ succ(u)` and `w ∈ pred(v)`) and add
   up `1 / log(degree(w))` for each. The intuition: a shared connector that is
   itself *rare* (low degree) is strong evidence, while a giant hub everyone
   touches is weak evidence — so we *downweight* high-degree connectors. (Analogy:
   two people who both know your reclusive aunt are probably connected; two
   people who both follow a celebrity are not.)
7. **`pref_attach`** — **preferential attachment**, here `out_deg(u) * in_deg(v)`.
   The "rich get richer" idea: a node that already links out a lot is more likely
   to form a *new* outgoing link, and a node that is already linked-to a lot is a
   likely target. Multiplying gives a single popularity score for the pair.
8. **`reciprocity_ind`** — a 0/1 flag: does the *reverse* edge `v -> u` exist in
   the train graph? Mutual links are common, so seeing `v -> u` strongly hints at
   `u -> v`.
9. **`path_len2`** — a 0/1 flag: is there a directed two-step path `u -> w -> v`?
   (Equivalently, is `succ(u) ∩ pred(v)` non-empty?)
10. **`path_len3_flag`** — a 0/1 flag: is there a directed path from `u` to `v` of
    length at most 3? (We cap the length to keep it cheap; long paths add little.)

We feed these to two standard classifiers:
- **Logistic Regression (LR)** — fits a weighted sum of the features and squashes
  it to a probability; simple, fast, interpretable.
- **Random Forest (RF)** — an ensemble of many decision "trees" that vote;
  captures non-linear interactions between features automatically.


In [8]:
# Precompute neighbour sets and degrees on the TRAIN graph (leakage-free).
succ   = {n: set(G_train.successors(n))   for n in G_train.nodes()}
pred   = {n: set(G_train.predecessors(n)) for n in G_train.nodes()}
indeg_t  = dict(G_train.in_degree())
outdeg_t = dict(G_train.out_degree())
train_edge_set = set(G_train.edges())

FEATURE_NAMES = ["src_out","src_in","tgt_out","tgt_in","common_succ","common_pred",
                 "dir_jaccard_succ","dir_jaccard_pred","dir_adamic_adar","pref_attach",
                 "reciprocity_ind","path_len2","path_len3_flag"]

def directed_adamic_adar(u, v):
    score = 0.0
    for w in succ.get(u, set()) & pred.get(v, set()):   # stepping stones u->w->v
        d = outdeg_t.get(w, 0) + indeg_t.get(w, 0)
        if d > 1:
            score += 1.0 / math.log(d)
    return score

def feats_for_pair(u, v):
    su, sv = succ.get(u, set()), succ.get(v, set())
    pu, pv = pred.get(u, set()), pred.get(v, set())
    cs = len(su & sv); cp = len(pu & pv)
    js = len(su & sv) / len(su | sv) if (su | sv) else 0.0
    jp = len(pu & pv) / len(pu | pv) if (pu | pv) else 0.0
    aa = directed_adamic_adar(u, v)
    pa = outdeg_t.get(u, 0) * indeg_t.get(v, 0)
    recip = 1.0 if (v, u) in train_edge_set else 0.0
    p2 = 1.0 if (su & pv) else 0.0
    if p2:
        p3 = 1.0
    else:
        p3 = 0.0
        for a in su:                       # length-3: u->a->b->v
            if succ.get(a, set()) & pv:
                p3 = 1.0; break
    return [outdeg_t.get(u,0), indeg_t.get(u,0), outdeg_t.get(v,0), indeg_t.get(v,0),
            cs, cp, js, jp, aa, pa, recip, p2, p3]

def build_matrix(pairs):
    return np.array([feats_for_pair(u, v) for (u, v) in pairs], dtype=float)

t0 = time.time()
Xtr = np.vstack([build_matrix(train_pos), build_matrix(neg_train)])
Xte = np.vstack([build_matrix(test_pos),  build_matrix(neg_test)])
ytr = np.concatenate([np.ones(len(train_pos)), np.zeros(len(neg_train))])
yte = np.concatenate([np.ones(len(test_pos)),  np.zeros(len(neg_test))])
print("topology features built in", round(time.time()-t0,1), "s")
print("X_train", Xtr.shape, "| X_test", Xte.shape)


topology features built in 2.6 s
X_train (78398, 13) | X_test (19598, 13)


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, roc_curve)

def evaluate(name, model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    proba = model.predict_proba(Xte)[:, 1]    # predicted probability of "link exists"
    pred  = (proba >= 0.5).astype(int)
    row = dict(model=name,
               AUC=roc_auc_score(yte, proba),
               accuracy=accuracy_score(yte, pred),
               precision=precision_score(yte, pred),
               recall=recall_score(yte, pred),
               F1=f1_score(yte, pred))
    fpr, tpr, _ = roc_curve(yte, proba)
    return row, (fpr, tpr)

# StandardScaler puts every feature on a comparable scale (mean 0, spread 1),
# which Logistic Regression needs to behave well.
base_lr = make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=1000, random_state=42))
base_rf = RandomForestClassifier(n_estimators=200, n_jobs=2, random_state=42)

results = []          # collect every model's metrics here
roc_curves = {}       # collect (fpr,tpr) for the ROC figure

r, roc = evaluate("Baseline LR (topology)", base_lr, Xtr, ytr, Xte, yte)
results.append(r); roc_curves[r["model"]] = roc
r, roc = evaluate("Baseline RF (topology)", base_rf, Xtr, ytr, Xte, yte)
results.append(r); roc_curves[r["model"]] = roc

pd.DataFrame(results).set_index("model").round(4)


,AUC,accuracy,precision,recall,F1
model,,,,,
Baseline LR (topology),0.9414,0.8726,0.8881,0.8527,0.8701
Baseline RF (topology),0.9417,0.8712,0.8738,0.8677,0.8708


In [10]:
# Which topology features matter most? Random Forest can tell us.
rf_imp = RandomForestClassifier(n_estimators=200, n_jobs=2, random_state=42).fit(Xtr, ytr)
imp = pd.Series(rf_imp.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
imp.iloc[::-1].plot.barh(ax=ax, color="#2ca02c")
ax.set_xlabel("Random-Forest importance (higher = more useful)")
ax.set_title("B1.4 Which directed topology features drive the baseline?")
fig.tight_layout()
na.save_fig(fig, "partB_feature_importance.png")
plt.close(fig)
print(imp.round(4).to_string())


dir_adamic_adar     0.2315
pref_attach         0.1395
path_len2           0.1078
dir_jaccard_pred    0.0986
dir_jaccard_succ    0.0704
common_pred         0.0614
tgt_in              0.0543
common_succ         0.0532
src_in              0.0500
tgt_out             0.0492
src_out             0.0489
reciprocity_ind     0.0241
path_len3_flag      0.0110


**How I solved this task (B1.4).**
I engineered eleven *direction-aware* features for each candidate pair —
in/out-degrees of both endpoints, counts and Jaccard overlaps of common
successors and predecessors, a directed Adamic-Adar score that rewards rare
shared connectors, preferential attachment from out- and in-degree, a flag for
whether the reverse edge exists, and flags for short directed paths of length 2
and ≤3 — computing all of them from the *train-only* graph. I trained both
Logistic Regression and a Random Forest. The baseline already predicts links
well (AUC around 0.94), and the importance plot shows that directed Adamic-Adar,
preferential attachment, and the length-2 path flag carry most of the signal —
i.e. "do `u` and `v` share a rare stepping-stone, are they both popular, and can
you already get from one to the other in two hops".

## B1.5 — Improved classifier with node2vec embeddings (5 pts)

### What is a node embedding? (plain words + analogy)
An **embedding** turns each node into a short list of numbers — a point in, say,
64-dimensional space — so that nodes which play *similar roles* in the graph end
up *near each other*. Think of it like placing every subreddit on a giant map
where "communities that get linked in the same contexts" sit close together. The
classifier can then ask geometric questions ("are these two points arranged like
a real edge?") instead of only hand-made graph counts.

### What node2vec does, step by step
**node2vec** learns these points using a "you are the company you keep" trick
borrowed from language models:
1. **Random walks.** Start at a node and take a stroll, repeatedly hopping to a
   neighbour at random, recording the sequence of nodes visited — like a sentence
   of words, but the "words" are subreddits. We do this many times from every
   node. On a *directed* graph the walker follows the arrow directions, so the
   walks capture directional structure.
2. **word2vec on the walks.** We feed these node-sequences to **word2vec**, an
   algorithm that learns a vector for each "word" so that words appearing in
   similar surroundings get similar vectors. Result: nodes that show up in
   similar walks get similar embeddings.

A tiny picture: if walks often read `... gaming -> leagueoflegends -> esports ...`
and also `... gaming -> dota2 -> esports ...`, then `leagueoflegends` and `dota2`
get placed near each other because they appear in the same neighbourhood.

We use modest settings so it runs fast on the shared CPU node:
`dimensions=64, walk_length=20, num_walks=10, workers=2`.

### Turning two node vectors into one *pair* feature
A classifier needs **one** feature row per pair, but we have **two** vectors
(`emb(u)` and `emb(v)`). Two standard ways to combine them:
- **Hadamard product** = multiply the vectors element-by-element. If a coordinate
  is large in *both* nodes it stays large; this highlights dimensions the two
  nodes *agree* on. It is the most common choice for link prediction.
- **Concatenation** = stick `emb(u)` then `emb(v)` into one long vector
  `[src | tgt]`. This keeps source and target separate, so the model can learn
  *direction* (it knows which half is the source).

We compare three "improved" setups: embeddings-only (Hadamard), embeddings-only
(concat), and the combination **topology features + embeddings**.


In [11]:
from node2vec import Node2Vec

print("Running node2vec on the directed TRAIN graph (modest params for shared CPU)...")
t0 = time.time()
n2v = Node2Vec(G_train, dimensions=64, walk_length=20, num_walks=10,
               weight_key="weight", workers=2, seed=42, quiet=True)
w2v = n2v.fit(window=5, min_count=1, batch_words=4, seed=42, workers=2)
emb = {n: w2v.wv[str(n)] for n in G_train.nodes()}
DIM = 64
zero = np.zeros(DIM)
print("node2vec finished in", round(time.time()-t0,1), "s ; embeddings:", len(emb), "x", DIM)


Running node2vec on the directed TRAIN graph (modest params for shared CPU)...


node2vec finished in 25.1 s ; embeddings: 1995 x 64


In [12]:
def pair_hadamard(pairs):
    return np.array([emb.get(u, zero) * emb.get(v, zero) for u, v in pairs], dtype=float)

def pair_concat(pairs):
    return np.array([np.concatenate([emb.get(u, zero), emb.get(v, zero)]) for u, v in pairs],
                    dtype=float)

# embeddings-only matrices
Xtr_h = np.vstack([pair_hadamard(train_pos), pair_hadamard(neg_train)])
Xte_h = np.vstack([pair_hadamard(test_pos),  pair_hadamard(neg_test)])
Xtr_c = np.vstack([pair_concat(train_pos),   pair_concat(neg_train)])
Xte_c = np.vstack([pair_concat(test_pos),    pair_concat(neg_test)])

# topology + embeddings (Hadamard) matrices
Xtr_combo = np.hstack([Xtr, Xtr_h])
Xte_combo = np.hstack([Xte, Xte_h])
print("Hadamard pair-features:", Xtr_h.shape, "| concat:", Xtr_c.shape,
      "| topo+emb:", Xtr_combo.shape)


Hadamard pair-features: (78398, 64) | concat: (78398, 128) | topo+emb: (78398, 77)


In [13]:
# (1) embeddings only, Hadamard, Logistic Regression
r, roc = evaluate("Emb-only Hadamard (LR)",
                  make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
                  Xtr_h, ytr, Xte_h, yte)
results.append(r); roc_curves[r["model"]] = roc

# (2) embeddings only, concatenation, Logistic Regression
r, roc = evaluate("Emb-only Concat (LR)",
                  make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
                  Xtr_c, ytr, Xte_c, yte)
results.append(r); roc_curves[r["model"]] = roc

# (3) embeddings only, Hadamard, Random Forest
r, roc = evaluate("Emb-only Hadamard (RF)",
                  RandomForestClassifier(n_estimators=200, n_jobs=2, random_state=42),
                  Xtr_h, ytr, Xte_h, yte)
results.append(r); roc_curves[r["model"]] = roc

# (4) topology + embeddings, Random Forest  <-- expected best
r, roc = evaluate("Topo+Emb Hadamard (RF)",
                  RandomForestClassifier(n_estimators=300, n_jobs=2, random_state=42),
                  Xtr_combo, ytr, Xte_combo, yte)
results.append(r); roc_curves[r["model"]] = roc

# (5) topology + embeddings, Logistic Regression
r, roc = evaluate("Topo+Emb Hadamard (LR)",
                  make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
                  Xtr_combo, ytr, Xte_combo, yte)
results.append(r); roc_curves[r["model"]] = roc

pd.DataFrame(results).set_index("model").round(4)


,AUC,accuracy,precision,recall,F1
model,,,,,
Baseline LR (topology),0.9414,0.8726,0.8881,0.8527,0.8701
Baseline RF (topology),0.9417,0.8712,0.8738,0.8677,0.8708
Emb-only Hadamard (LR),0.8023,0.7234,0.7497,0.6707,0.7080
Emb-only Concat (LR),0.7444,0.6804,0.6806,0.6800,0.6803
Emb-only Hadamard (RF),0.8778,0.7958,0.8219,0.7553,0.7872
Topo+Emb Hadamard (RF),0.9493,0.8823,0.8932,0.8685,0.8806
Topo+Emb Hadamard (LR),0.9439,0.8717,0.8994,0.8369,0.8671


**How I solved this task (B1.5).**
I ran node2vec on the directed train graph to learn a 64-number vector for each
subreddit, using short weighted random walks plus word2vec so that
similarly-positioned communities land near each other. I converted each pair of
node vectors into one pair-feature in two standard ways — the element-wise
Hadamard product and the source||target concatenation — and trained classifiers
on embeddings alone and on embeddings combined with the topology features.
Embeddings alone are good but a notch below the hand-made topology baseline
(AUC ~0.88 vs ~0.94); the **combination** of topology + embeddings is the best
model overall (AUC ~0.95), because the two describe complementary things: the
topology counts capture exact local overlap, while the embeddings capture softer
"these communities behave alike" structure.

## B1.6 — Evaluation: AUC, accuracy, precision, recall, F1, ROC curves (3 pts)

We score every model with five numbers. Defining each in plain words:

- **Accuracy** — fraction of test pairs labelled correctly. Easy to read, but it
  can hide problems on imbalanced data (ours is balanced, so it is meaningful
  here).
- **Precision** — of the pairs the model *claims* are links, what fraction really
  are? High precision = few false alarms.
- **Recall** — of the real links, what fraction did the model *catch*? High
  recall = few misses.
- **F1** — the balanced average (harmonic mean) of precision and recall, a single
  number that is high only when *both* are high.
- **AUC (Area Under the ROC Curve)** — the headline metric for link prediction.
  Imagine picking one real link and one non-link at random; AUC is the
  probability the model gives the real link the *higher* score. **AUC = 1.0** is
  perfect, **AUC = 0.5** is coin-flipping. It is threshold-free, so it judges the
  ranking quality regardless of where we draw the 0.5 cutoff.

An **ROC curve** plots, as we sweep the decision threshold from strict to
lenient, the **true-positive rate** (recall) on the y-axis against the
**false-positive rate** (share of non-links wrongly flagged) on the x-axis. A
curve that hugs the top-left corner is excellent; the diagonal line is random
guessing. The AUC is literally the area under this curve.


In [14]:
# ROC curves for the headline models.
fig, ax = plt.subplots(figsize=(7.2, 6.0))
show = ["Baseline LR (topology)", "Baseline RF (topology)",
        "Emb-only Hadamard (RF)", "Topo+Emb Hadamard (RF)"]
auc_lookup = {r["model"]: r["AUC"] for r in results}
for name in show:
    fpr, tpr = roc_curves[name]
    ax.plot(fpr, tpr, lw=2, label=f"{name}  (AUC={auc_lookup[name]:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="random guess (AUC=0.5)")
ax.set_xlabel("False-positive rate (non-links wrongly flagged)")
ax.set_ylabel("True-positive rate / recall (real links caught)")
ax.set_title("B1.6 ROC curves — random-split directed link prediction")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
na.save_fig(fig, "partB_roc_curves.png")
plt.close(fig)
print("saved figures/partB_roc_curves.png")
pd.DataFrame(results).set_index("model").round(4)


saved figures/partB_roc_curves.png


,AUC,accuracy,precision,recall,F1
model,,,,,
Baseline LR (topology),0.9414,0.8726,0.8881,0.8527,0.8701
Baseline RF (topology),0.9417,0.8712,0.8738,0.8677,0.8708
Emb-only Hadamard (LR),0.8023,0.7234,0.7497,0.6707,0.7080
Emb-only Concat (LR),0.7444,0.6804,0.6806,0.6800,0.6803
Emb-only Hadamard (RF),0.8778,0.7958,0.8219,0.7553,0.7872
Topo+Emb Hadamard (RF),0.9493,0.8823,0.8932,0.8685,0.8806
Topo+Emb Hadamard (LR),0.9439,0.8717,0.8994,0.8369,0.8671


**How I solved this task (B1.6).**
I evaluated every model on the held-out 20% test set using AUC, accuracy,
precision, recall and F1, and drew ROC curves for the headline models. AUC is the
metric I trust most because it judges how well each model *ranks* real links above
non-links regardless of the cutoff. The curves all bow well toward the top-left
(far above the random diagonal), and the topology+embeddings Random Forest sits
highest, confirming it is the strongest ranker.

## B1.7 — Comparison: baseline vs improved (2 pts)

In [15]:
tbl = pd.DataFrame(results).set_index("model").round(4)
best_base = tbl.loc[["Baseline LR (topology)", "Baseline RF (topology)"], "AUC"].max()
best_impr = tbl.loc[["Topo+Emb Hadamard (RF)", "Topo+Emb Hadamard (LR)"], "AUC"].max()
print("Best baseline AUC (topology only):", round(best_base, 4))
print("Best improved AUC (topology+emb) :", round(best_impr, 4))
print("Absolute AUC gain from embeddings:", round(best_impr - best_base, 4))
tbl


Best baseline AUC (topology only): 0.9417
Best improved AUC (topology+emb) : 0.9493
Absolute AUC gain from embeddings: 0.0076


,AUC,accuracy,precision,recall,F1
model,,,,,
Baseline LR (topology),0.9414,0.8726,0.8881,0.8527,0.8701
Baseline RF (topology),0.9417,0.8712,0.8738,0.8677,0.8708
Emb-only Hadamard (LR),0.8023,0.7234,0.7497,0.6707,0.7080
Emb-only Concat (LR),0.7444,0.6804,0.6806,0.6800,0.6803
Emb-only Hadamard (RF),0.8778,0.7958,0.8219,0.7553,0.7872
Topo+Emb Hadamard (RF),0.9493,0.8823,0.8932,0.8685,0.8806
Topo+Emb Hadamard (LR),0.9439,0.8717,0.8994,0.8369,0.8671


**Discussion (baseline vs improved).**

- The **baseline** (directed topology features only) is already strong: AUC
  around **0.94**. This tells us most of the predictive signal lives in simple,
  interpretable structure — shared rare connectors (Adamic-Adar), joint
  popularity (preferential attachment), and short directed paths.
- **Embeddings alone** reach AUC around **0.88** — respectable, and impressive
  given they use *no* hand-crafted graph counts, only learned geometry. They sit
  slightly below the baseline because the random walks blur some of the exact
  local overlap that the topology counts measure precisely.
- The **improved combined model** (topology + node2vec, Random Forest) is the
  best, AUC around **0.95**. The gain over the baseline is small in absolute terms
  but consistent, and it is the expected outcome: embeddings add *complementary*
  "soft similarity" information on top of the exact counts, so together they edge
  out either one alone.

**Takeaway.** For this network, hand-made directed topology features are a very
strong, cheap baseline; node2vec embeddings give a modest, reliable extra lift
when combined with them, rather than replacing them.

**How I solved this task (B1.7).**
I assembled all models into one metrics table and computed the AUC gap between
the best topology-only baseline and the best topology+embedding model. The
comparison shows embeddings provide a small but consistent improvement on top of
an already-strong topology baseline, and that combining the two beats either in
isolation.

## Bonus — Future Link Prediction with a temporal split (+5 pts)

The random split above mixes past and future edges freely. A tougher, more
realistic test is **temporal**: stand at a moment in time, learn only from what
happened **before** it, then predict links that appear **after** it.

### The setup, in plain words
1. **Pick a cutoff time `T`** — we use the 80th percentile of edges' *first*
   appearance, so 80% of links (by first-appearance) are "past" and 20% are
   "future".
2. **Past graph.** Build a directed graph from only the edges whose first
   appearance is before `T`. All features come from this past-only graph.
3. **Positive future links.** A pair `u -> v` counts as a positive if its *first*
   appearance is at or after `T`, both `u` and `v` already existed before `T`
   (we only judge links between *known* communities), and the edge did **not**
   exist before `T` (it is genuinely new).
4. **Negative future links.** Pairs of existing nodes that *never* become an edge
   (same WCC filter as before), balanced 1:1 with the positives.
5. **Two predictors to compare:**
   - **(a) Random** — assign each test pair a random score. By definition this
     gives AUC ≈ 0.5 and is our "do-nothing" reference.
   - **(b) Informed** — our classifier (topology features, and topology+node2vec
     embeddings) trained on the *past* graph, scoring the future pairs.

### Why future prediction is harder than the random split
In the random split, when we score a test edge `u -> v`, the graph still contains
*many other* edges from the same time period that surround it, so the local
neighbourhood is rich. In the temporal split, future positives are by
construction **brand-new** links — at cutoff time their endpoints were *not yet*
connected through that link, so the past graph gives a thinner, weaker signal.
We therefore expect the informed AUC to drop somewhat below the ~0.95 we saw on
the random split, while still beating random by a wide margin.


In [16]:
# Cutoff at the 80th percentile of first-appearance times.
T = agg["first"].quantile(0.80)
past   = agg[agg["first"] <  T]
future = agg[agg["first"] >= T]
print("Cutoff T:", T)
print("past edges:", len(past), "| future edges:", len(future))

# Past-only directed graph; all bonus features come from here.
Gp = nx.from_pandas_edgelist(past, "SOURCE_SUBREDDIT", "TARGET_SUBREDDIT",
                             edge_attr=["weight"], create_using=nx.DiGraph)
pre_nodes = set(Gp.nodes()); pre_edges = set(Gp.edges())
all_edge_set = set(zip(agg["SOURCE_SUBREDDIT"], agg["TARGET_SUBREDDIT"]))
print("past graph:", Gp.number_of_nodes(), "nodes,", Gp.number_of_edges(), "edges")

# Positive future links: new edges among already-existing nodes.
fut_pos = [(u, v) for u, v in zip(future["SOURCE_SUBREDDIT"], future["TARGET_SUBREDDIT"])
           if u in pre_nodes and v in pre_nodes and (u, v) not in pre_edges]
print("future positives (new links among existing nodes):", len(fut_pos))


Cutoff T: 2016-07-01 21:44:47.200000
past edges: 39198 | future edges: 9800
past graph: 1948 nodes, 39198 edges
future positives (new links among existing nodes): 8915


In [17]:
# Negative future links: existing-node pairs that never become edges, same WCC.
wcc_of_p = {}
for ci, comp in enumerate(nx.weakly_connected_components(Gp)):
    for nd in comp:
        wcc_of_p[nd] = ci
arr = np.array(list(pre_nodes), dtype=object)
rb = np.random.RandomState(42)
fut_neg = set(); need = len(fut_pos)
while len(fut_neg) < need:
    us = arr[rb.randint(0, len(arr), size=need*2)]
    vs = arr[rb.randint(0, len(arr), size=need*2)]
    for u, v in zip(us, vs):
        if u == v or (u, v) in all_edge_set or (u, v) in fut_neg: continue
        if wcc_of_p.get(u) != wcc_of_p.get(v): continue
        fut_neg.add((u, v))
        if len(fut_neg) >= need: break
fut_neg = list(fut_neg)
print("future negatives:", len(fut_neg))


future negatives: 8915


In [18]:
# Features computed from the PAST graph only.
succ_p = {n: set(Gp.successors(n))   for n in Gp.nodes()}
pred_p = {n: set(Gp.predecessors(n)) for n in Gp.nodes()}
indeg_p  = dict(Gp.in_degree()); outdeg_p = dict(Gp.out_degree())

def aa_p(u, v):
    s = 0.0
    for w in succ_p.get(u, set()) & pred_p.get(v, set()):
        d = outdeg_p.get(w, 0) + indeg_p.get(w, 0)
        if d > 1: s += 1.0/math.log(d)
    return s

def feats_p(u, v):
    su, sv = succ_p.get(u, set()), succ_p.get(v, set())
    pu, pv = pred_p.get(u, set()), pred_p.get(v, set())
    cs = len(su & sv); cp = len(pu & pv)
    js = len(su & sv)/len(su | sv) if (su | sv) else 0.0
    jp = len(pu & pv)/len(pu | pv) if (pu | pv) else 0.0
    aa = aa_p(u, v); pa = outdeg_p.get(u,0)*indeg_p.get(v,0)
    recip = 1.0 if (v, u) in pre_edges else 0.0
    p2 = 1.0 if (su & pv) else 0.0
    if p2: p3 = 1.0
    else:
        p3 = 0.0
        for a in su:
            if succ_p.get(a, set()) & pv: p3 = 1.0; break
    return [outdeg_p.get(u,0), indeg_p.get(u,0), outdeg_p.get(v,0), indeg_p.get(v,0),
            cs, cp, js, jp, aa, pa, recip, p2, p3]

# node2vec on the PAST graph.
print("node2vec on past graph...")
n2v_p = Node2Vec(Gp, dimensions=64, walk_length=20, num_walks=10,
                 weight_key="weight", workers=2, seed=42, quiet=True)
w2v_p = n2v_p.fit(window=5, min_count=1, batch_words=4, seed=42, workers=2)
emb_p = {n: w2v_p.wv[str(n)] for n in Gp.nodes()}
zero_p = np.zeros(64)
def had_p(pairs):
    return np.array([emb_p.get(u, zero_p)*emb_p.get(v, zero_p) for u, v in pairs], dtype=float)
print("done")


node2vec on past graph...


done


In [19]:
# Build a TRAIN set from the past graph (predict past edges), then test on the future.
past_edges_list = list(pre_edges)
r2 = np.random.RandomState(7)
samp = r2.choice(len(past_edges_list), size=min(20000, len(past_edges_list)), replace=False)
train_pos_p = [past_edges_list[i] for i in samp]
tr_neg_p = set()
while len(tr_neg_p) < len(train_pos_p):
    us = arr[r2.randint(0, len(arr), size=len(train_pos_p))]
    vs = arr[r2.randint(0, len(arr), size=len(train_pos_p))]
    for u, v in zip(us, vs):
        if u == v or (u, v) in all_edge_set or (u, v) in tr_neg_p: continue
        if wcc_of_p.get(u) != wcc_of_p.get(v): continue
        tr_neg_p.add((u, v))
        if len(tr_neg_p) >= len(train_pos_p): break
tr_neg_p = list(tr_neg_p)

Xtr_topo_p = np.vstack([np.array([feats_p(u,v) for u,v in train_pos_p]),
                        np.array([feats_p(u,v) for u,v in tr_neg_p])])
ytr_p = np.concatenate([np.ones(len(train_pos_p)), np.zeros(len(tr_neg_p))])
Xte_topo_p = np.vstack([np.array([feats_p(u,v) for u,v in fut_pos]),
                        np.array([feats_p(u,v) for u,v in fut_neg])])
yte_p = np.concatenate([np.ones(len(fut_pos)), np.zeros(len(fut_neg))])

Xtr_combo_p = np.hstack([Xtr_topo_p, np.vstack([had_p(train_pos_p), had_p(tr_neg_p)])])
Xte_combo_p = np.hstack([Xte_topo_p, np.vstack([had_p(fut_pos),     had_p(fut_neg)])])
print("future test set:", Xte_topo_p.shape, "(half positive, half negative)")


future test set: (17830, 13) (half positive, half negative)


In [20]:
# (a) RANDOM baseline
rnd_scores = np.random.RandomState(123).rand(len(yte_p))
auc_random = roc_auc_score(yte_p, rnd_scores)

# (b) informed: topology only
rf_t = RandomForestClassifier(n_estimators=200, n_jobs=2, random_state=42).fit(Xtr_topo_p, ytr_p)
proba_t = rf_t.predict_proba(Xte_topo_p)[:, 1]
auc_topo = roc_auc_score(yte_p, proba_t)

# (b) informed: topology + embeddings
rf_c = RandomForestClassifier(n_estimators=300, n_jobs=2, random_state=42).fit(Xtr_combo_p, ytr_p)
proba_c = rf_c.predict_proba(Xte_combo_p)[:, 1]
auc_combo = roc_auc_score(yte_p, proba_c)

print("=== BONUS: future link prediction AUCs ===")
print(f"  (a) Random predictor          : {auc_random:.4f}")
print(f"  (b) Informed topology (RF)    : {auc_topo:.4f}")
print(f"  (b) Informed topo+emb (RF)    : {auc_combo:.4f}")


=== BONUS: future link prediction AUCs ===
  (a) Random predictor          : 0.4962
  (b) Informed topology (RF)    : 0.9038
  (b) Informed topo+emb (RF)    : 0.9142


In [21]:
# ROC curves for the bonus: random vs informed.
from sklearn.metrics import roc_curve as _roc
fig, ax = plt.subplots(figsize=(7.2, 6.0))
for scores, lab, auc in [(rnd_scores, "Random predictor", auc_random),
                         (proba_t, "Informed topology (RF)", auc_topo),
                         (proba_c, "Informed topo+emb (RF)", auc_combo)]:
    fpr, tpr, _ = _roc(yte_p, scores)
    ax.plot(fpr, tpr, lw=2, label=f"{lab} (AUC={auc:.3f})")
ax.plot([0,1],[0,1],"k--",lw=1)
ax.set_xlabel("False-positive rate"); ax.set_ylabel("True-positive rate")
ax.set_title("Bonus ROC — future link prediction (temporal split)")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
na.save_fig(fig, "partB_bonus_future_roc.png")
plt.close(fig)
print("saved figures/partB_bonus_future_roc.png")


saved figures/partB_bonus_future_roc.png


**Bonus discussion — does the classifier generalize to the future?**

Yes, clearly. The **random predictor scores AUC ≈ 0.50** (a coin flip, exactly
as expected), while the **informed classifier scores AUC ≈ 0.90+** on genuinely
new future links it has never seen. So the structural patterns learned from the
past — shared rare connectors, joint popularity, short directed paths, and
embedding geometry — really do carry forward and let us anticipate which
communities will start linking next.

**Why this is harder than the random split (and why the AUC is a bit lower).**
On the random split we reached ~0.95; here we land a little lower. The reason is
baked into the setup: future positives are *brand-new* edges, so at cutoff time
their two endpoints were not yet joined through that link, and the past graph
offers a thinner signal than the rich same-period neighbourhood available in the
random split. The network also *drifts* over time — new fads, new communities,
changing interests — so a model fit on the past is always chasing a slightly
moving target. The gap between ~0.95 (random split) and ~0.90 (temporal) is
exactly this "predicting the future is harder than filling in the present" effect.

**How I solved this task (Bonus).**
I chose a cutoff at the 80th percentile of first-appearance times, built a
past-only directed graph, and defined positives as new links that first appear
after the cutoff between communities that already existed before it, with balanced
non-edge negatives. I computed all features (directed topology + node2vec) from
the past graph only, trained the classifier to predict past edges, and tested it
on the future links. Comparing against a random-score baseline, the informed
model's AUC (~0.90+) towers over random (~0.50), demonstrating real
generalization to the future, while sitting modestly below the random-split AUC
because forecasting brand-new links is inherently harder.

## Summary of results

| Setting | Model | AUC |
|---|---|---|
| Random split | Baseline (directed topology) | ~0.94 |
| Random split | Embeddings only (node2vec) | ~0.88 |
| Random split | **Topology + embeddings (best)** | **~0.95** |
| Temporal (bonus) | Random predictor | ~0.50 |
| Temporal (bonus) | Informed (topology) | ~0.90 |
| Temporal (bonus) | Informed (topology + embeddings) | ~0.91 |

**Libraries used:** networkx (graph + features), node2vec + gensim (embeddings),
scikit-learn (classifiers + metrics), pandas/numpy (data), matplotlib (figures).

**Data source:** SNAP Reddit Hyperlink Network (body),
https://snap.stanford.edu/data/soc-RedditHyperlinks.html

**Reproducibility:** all seeds fixed to 42; sampling = subgraph induced by the
top-2000 most active subreddits; node2vec params dimensions=64, walk_length=20,
num_walks=10, workers=2.
